**Camada Silver**

A Camada Silver entrega dados limpos, padronizados e tipados corretamente, permitindo integração confiável entre tabelas e servindo como base sólida para a construção das tabelas Fato e Dimensão na Camada Gold.

In [0]:
from pyspark.sql import functions as F

# ===============================================================
# 1) CARREGAR TABELAS BRONZE
# ===============================================================

bronze_full         = spark.table("bronze_full")
bronze_cartoes      = spark.table("bronze_cartoes")
bronze_gols         = spark.table("bronze_gols")
bronze_estatisticas = spark.table("bronze_estatisticas")

print("Tabelas Bronze carregadas.")


# ===============================================================
# 2) SILVER FULL — PADRONIZAÇÃO DE PARTIDAS
# ===============================================================

silver_full = (
    bronze_full

        # Conversão segura da coluna de data
        .withColumn("data_dt", F.to_date("data", "dd/MM/yyyy"))

        # Extrai somente valores válidos de HH:mm
        .withColumn(
            "hora_limpa",
            F.regexp_extract("hora", r"([0-9]{2}:[0-9]{2})", 1)
        )

        # Constrói timestamp somente quando a hora for válida
        .withColumn(
            "data_hora_ts",
            F.expr("try_to_timestamp(concat(data, ' ', hora_limpa), 'dd/MM/yyyy HH:mm')")
        )

        # Conversões numéricas essenciais
        .withColumn("partida_id", F.col("id").cast("int"))
        .withColumn("mandante_placar", F.col("mandante_placar").cast("int"))
        .withColumn("visitante_placar", F.col("visitante_placar").cast("int"))

        # Normalização textual robusta
        .withColumn(
            "mandante",
            F.initcap(F.trim(F.regexp_replace("mandante", r"[\u00A0\u2007\u202F]", "")))
        )
        .withColumn(
            "visitante",
            F.initcap(F.trim(F.regexp_replace("visitante", r"[\u00A0\u2007\u202F]", "")))
        )
        .withColumn(
            "arena",
            F.initcap(F.trim(F.regexp_replace("arena", r"[\u00A0\u2007\u202F]", "")))
        )

        # Correção solicitada: "-" → "Empate"
        .withColumn(
            "vencedor",
            F.when(F.col("vencedor") == "-", "Empate")
             .otherwise(F.col("vencedor"))
        )
)

silver_full.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver_full")

print("Tabela criada: silver_full")


# ===============================================================
# 3) SILVER GOLS — TRATAMENTO DE ACRESCIMOS
# ===============================================================

silver_gols = (
    bronze_gols
        .withColumn(
            "clube",
            F.initcap(F.trim(F.regexp_replace("clube", r"[\u00A0\u2007\u202F]", "")))
        )
        .withColumn("partida_id", F.col("partida_id").cast("int"))
        .withColumn("rodata", F.col("rodata").cast("int"))

        # Minuto + acréscimo
        .withColumn(
            "minuto_int",
            F.when(
                F.col("minuto").contains("+"),
                F.split("minuto", "\\+").getItem(0).cast("int") +
                F.split("minuto", "\\+").getItem(1).cast("int")
            )
            .otherwise(F.col("minuto").cast("int"))
        )
)

silver_gols.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver_gols")

print("Tabela criada: silver_gols")


# ===============================================================
# 4) SILVER CARTÕES — PADRONIZAÇÃO + CORREÇÃO “Zagueira”
# ===============================================================

silver_cartoes = (
    bronze_cartoes

        .withColumn(
            "clube",
            F.initcap(F.trim(F.regexp_replace("clube", r"[\u00A0\u2007\u202F]", "")))
        )

        # Correção solicitada: "Zagueira" → "Zagueiro"
        .withColumn(
            "posicao",
            F.when(F.col("posicao") == "Zagueira", "Zagueiro")
             .otherwise(F.col("posicao"))
        )

        .withColumn("partida_id", F.col("partida_id").cast("int"))
        .withColumn("rodata", F.col("rodata").cast("int"))
        .withColumn("num_camisa", F.col("num_camisa").cast("int"))

        # Minuto tratado
        .withColumn(
            "minuto_int",
            F.when(
                F.col("minuto").contains("+"),
                F.split("minuto", "\\+").getItem(0).cast("int") +
                F.split("minuto", "\\+").getItem(1).cast("int")
            )
            .otherwise(F.col("minuto").cast("int"))
        )
)

silver_cartoes.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver_cartoes")

print("Tabela criada: silver_cartoes")


# ===============================================================
# 5) SILVER ESTATÍSTICAS — SEM ALTERAR PERCENTUAIS
# ===============================================================

silver_estatisticas = (
    bronze_estatisticas

        .withColumn(
            "clube",
            F.initcap(F.trim(F.regexp_replace("clube", r"[\u00A0\u2007\u202F]", "")))
        )
        .withColumn("partida_id", F.col("partida_id").cast("int"))
        .withColumn("rodata", F.col("rodata").cast("int"))

        # Conversões numéricas básicas
        .withColumn("chutes", F.col("chutes").cast("int"))
        .withColumn("chutes_no_alvo", F.col("chutes_no_alvo").cast("int"))
        .withColumn("passes", F.col("passes").cast("int"))
        .withColumn("faltas", F.col("faltas").cast("int"))
        .withColumn("cartao_amarelo", F.col("cartao_amarelo").cast("int"))
        .withColumn("cartao_vermelho", F.col("cartao_vermelho").cast("int"))
        .withColumn("impedimentos", F.col("impedimentos").cast("int"))
        .withColumn("escanteios", F.col("escanteios").cast("int"))

        # Percentuais: apenas converte quando formato válido XX%
        .withColumn(
            "posse_de_bola_num",
            F.when(F.col("posse_de_bola").rlike("^[0-9]+%$"),
                   F.regexp_replace("posse_de_bola", "%", "").cast("double"))
        )

        .withColumn(
            "precisao_passes_num",
            F.when(F.col("precisao_passes").rlike("^[0-9]+%$"),
                   F.regexp_replace("precisao_passes", "%", "").cast("double"))
        )
)

silver_estatisticas.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver_estatisticas")

print("Tabela criada: silver_estatisticas")

print("\n✔ ETAPA SILVER COMPLETA\n")


Tabelas Bronze carregadas.
Tabela criada: silver_full
Tabela criada: silver_gols
Tabela criada: silver_cartoes
Tabela criada: silver_estatisticas

✔ ETAPA SILVER COMPLETA

